## MNIST

In [ ]:
from sklearn.datasets import fetch_openml

import numpy as np

import holoviews as hv
hv.extension("bokeh")

In [ ]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False, cache=True)

X, Y = mnist.data, mnist.target.astype(int)
X, Y 

In [ ]:
def as_one_hot(arr: np.ndarray):
    return np.eye(np.max(arr + 1))[arr]

Data preparations: x is scaled down to prevent overflows, y is converted from [..., 0, 2, 1, ...] to [..., [1, 0, 0], [0, 0, 1], [0, 1, 0], ...] and converted to floats

In [ ]:
X = X.astype(np.float32) / 255.0
Y = as_one_hot(Y).astype(np.float32)

In [ ]:
indices = np.random.permutation(X.shape[0])
split_point = int(X.shape[0] * 0.7)
train_indices = indices[:split_point]
test_indices = indices[split_point:]

X_train, Y_train = X[train_indices], Y[train_indices]
X_test, Y_test = X[test_indices], Y[test_indices]

In [ ]:
def our_api():
    from model import Sequential
    from layer import Dense, ReLU, SoftMax
    from optimizer import SGDOptimizer
    from loss import SumOfSquares
    
    model = Sequential([
        Dense(784, 250),
        ReLU(),
        Dense(250, 250),
        ReLU(),
        Dense(250, 10),
        SoftMax()
    ])

    model.compile_(optimizer=SGDOptimizer(lr=0.01, momentum=0.9), loss=SumOfSquares())

    hist = model.fit(X_train, Y_train, epochs=300, batch_size=100)
    return hist

In [ ]:
def torch_api():
    import torch
    from torch.nn import Sequential, Linear, ReLU, Softmax, MSELoss
    from torch.optim import SGD
    
    model = Sequential(
        Linear(784, 250), 
        ReLU(), 
        Linear(250, 250), 
        ReLU(), 
        Linear(250, 10), 
        Softmax(dim=1)
    )
    optimizer = SGD(model.parameters(), lr=0.01, momentum=0.9)
    loss = MSELoss()

    epochs = 300
    batch_size = 100

    n = X_train.shape[0]

    hist = {"loss": []}

    for epoch in range(epochs):
        idx = np.arange(n)
        np.random.shuffle(idx)

        for batch_left in range(0, n, batch_size):
            batch_idxs = idx[batch_left:(batch_left + batch_size)]
            x_batch = torch.from_numpy(X_train[batch_idxs].astype(np.float32))
            y_batch = torch.from_numpy(Y_train[batch_idxs].astype(np.float32))

            optimizer.zero_grad()
            preds = model(x_batch)

            loss_ = loss(preds, y_batch)
            loss_.backward()

            optimizer.step()
        
        print(f"epoch {epoch}: loss = {loss_.item():.4f}")
        hist["loss"].append(loss_.item())

    return hist

Our api vs torch api

In [ ]:
hist1 = our_api()

In [ ]:
hist2 = torch_api()

In [ ]:
hv.Curve(hist1["loss"]) * hv.Curve(hist2["loss"]).opts(width=600, height=600)